# Control Ratio — Descriptive and Statistical Analysis

This notebook analyzes control-ratio experiment results from `control_ratio_fold_results.csv`.

**Analyses performed**
1. Validate the data structure, number of observations, and fold pairs.
2. Compute descriptive statistics for PR-AUC (`pr_auc_ap`), MCC (`mcc`), and Balanced Accuracy (`balanced_accuracy`).
3. Perform the Friedman test to compare the five ratio conditions for each model and metric.
4. Perform pairwise Wilcoxon signed-rank tests between ratios, applying the Holm correction within each model–metric combination.
5. Create a three-panel line chart, with one panel per metric, ratios on the x-axis, and models represented by separate lines, following the style of the provided example.
6. Export tables as CSV files and figures as PNG/PDF files.

**Important:** The observational unit used is the `repeat`–`fold` pair within the same model. Repeated cross-validation produces folds that are not fully independent; therefore, p-values should be interpreted cautiously and reported as analyses of repeated-CV folds, not as evidence that the observations are independent.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
from itertools import combinations
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# Adjust this if the file name or path differs.
INPUT_FILE = Path("./control_ratio_fold_results.csv")
OUTPUT_DIR = Path("control_ratio_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = {
    "pr_auc_ap": "PR-AUC",
    "mcc": "MCC",
    "balanced_accuracy": "Balanced Accuracy",
}

RATIO_ORDER = ["original", "1:1", "1.5:1", "2:1", "5:1"]
RATIO_LABELS = {
    "original": "Original",
    "1:1": "1:1",
    "1.5:1": "1.5:1",
    "2:1": "2:1",
    "5:1": "5:1",
}

ALPHA = 0.05
RANDOM_STATE = 42

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 15,
})

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {INPUT_FILE.resolve()}\n"
        "Place the CSV file in the notebook's working directory or update INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE)
print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())

## 2. Data validation and cleaning

In [ ]:
required_cols = [
    "sampler", "ratio_name", "model", "repeat", "fold", *METRICS.keys()
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Required columns not found: {missing}")

# Use only the required columns; metric values are not modified.
df["ratio_name"] = df["ratio_name"].astype(str).str.strip()
df["model"] = df["model"].astype(str).str.strip()
df["sampler"] = df["sampler"].astype(str).str.strip()

for col in ["repeat", "fold", *METRICS.keys()]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["repeat", "fold", *METRICS.keys()]).copy()
df["repeat"] = df["repeat"].astype(int)
df["fold"] = df["fold"].astype(int)

# Check the ratios actually present; do not assume unavailable levels.
observed_ratios = df["ratio_name"].dropna().unique().tolist()
unknown_ratios = [r for r in observed_ratios if r not in RATIO_ORDER]
if unknown_ratios:
    print("Warning: ratio_name values were found outside the configured order:", unknown_ratios)

ratio_order_present = [r for r in RATIO_ORDER if r in observed_ratios]
ratio_order_present += [r for r in observed_ratios if r not in ratio_order_present]

models = sorted(df["model"].unique().tolist())
samplers = sorted(df["sampler"].unique().tolist())

print("Sampler:", samplers)
print("Model:", models)
print("Ratios:", ratio_order_present)
print("Number of rows after validation:", len(df))

# Duplicate keys may make pairing ambiguous. Stop rather than silently aggregating.
key_cols = ["sampler", "model", "ratio_name", "repeat", "fold"]
duplicate_mask = df.duplicated(subset=key_cols, keep=False)
if duplicate_mask.any():
    display(df.loc[duplicate_mask, key_cols].sort_values(key_cols).head(30))
    raise ValueError(
        "Duplicate keys were found for sampler/model/ratio/repeat/fold. "
        "Check the source file before running the statistical tests."
    )

# Audit the number of observations per condition
observation_audit = (
    df.groupby(["sampler", "model", "ratio_name"], dropna=False)
      .agg(n_rows=("fold", "size"),
           n_repeats=("repeat", "nunique"),
           n_folds=("fold", "nunique"))
      .reset_index()
)
observation_audit.to_csv(OUTPUT_DIR / "01_observation_audit.csv", index=False)
display(observation_audit)

## 3. Descriptive statistics

In [ ]:
descriptive = (
    df.groupby(["sampler", "model", "ratio_name"], dropna=False)[list(METRICS.keys())]
      .agg(["count", "mean", "std", "median", "min", "max"])
)
descriptive.columns = [f"{metric}_{stat}" for metric, stat in descriptive.columns]
descriptive = descriptive.reset_index()

# Sort ratios according to the conceptual order
ratio_rank = {ratio: i for i, ratio in enumerate(ratio_order_present)}
descriptive["_ratio_order"] = descriptive["ratio_name"].map(ratio_rank)
descriptive = descriptive.sort_values(["sampler", "model", "_ratio_order"]).drop(columns="_ratio_order")
descriptive.to_csv(OUTPUT_DIR / "02_descriptive_statistics.csv", index=False)

display(descriptive)

## 4. Friedman test

The Friedman test is run separately for each sampler, model, and metric combination. The five ratios are treated as repeated conditions for the same `repeat`–`fold` pairs. If any ratio is missing for a pair, the analysis for that combination uses only the available complete pairs; the actual number of pairs is reported.

In [ ]:
friedman_rows = []

for sampler in samplers:
    for model in models:
        sub = df[(df["sampler"] == sampler) & (df["model"] == model)].copy()

        for metric, metric_label in METRICS.items():
            wide = sub.pivot_table(
                index=["repeat", "fold"],
                columns="ratio_name",
                values=metric,
                aggfunc="first",
            )

            available_ratios = [r for r in ratio_order_present if r in wide.columns]
            complete = wide[available_ratios].dropna()

            if len(available_ratios) < 3 or len(complete) < 2:
                friedman_rows.append({
                    "sampler": sampler,
                    "model": model,
                    "metric": metric,
                    "metric_label": metric_label,
                    "ratios_tested": "; ".join(available_ratios),
                    "n_complete_pairs": len(complete),
                    "friedman_statistic": np.nan,
                    "p_value": np.nan,
                    "status": "Insufficient ratios or complete pairs",
                })
                continue

            arrays = [complete[r].to_numpy() for r in available_ratios]
            # If all conditions are exactly identical, SciPy may produce a degenerate case.
            try:
                stat, pval = friedmanchisquare(*arrays)
                status = "ok"
            except Exception as exc:
                stat, pval = np.nan, np.nan
                status = f"error: {type(exc).__name__}: {exc}"

            friedman_rows.append({
                "sampler": sampler,
                "model": model,
                "metric": metric,
                "metric_label": metric_label,
                "ratios_tested": "; ".join(available_ratios),
                "n_complete_pairs": len(complete),
                "friedman_statistic": stat,
                "p_value": pval,
                "status": status,
            })

friedman_results = pd.DataFrame(friedman_rows)
friedman_results.to_csv(OUTPUT_DIR / "03_friedman_tests.csv", index=False)
display(friedman_results)

## 5. Wilcoxon signed-rank test and Holm correction

The Wilcoxon test is performed for each ratio pair within the same model and metric, using the same `repeat`–`fold` pairs. The Holm correction is applied to all pairwise ratio comparisons within each sampler–model–metric combination. The case in which all paired differences are zero is handled separately.

In [ ]:
wilcoxon_rows = []

for sampler in samplers:
    for model in models:
        sub = df[(df["sampler"] == sampler) & (df["model"] == model)].copy()

        for metric, metric_label in METRICS.items():
            wide = sub.pivot_table(
                index=["repeat", "fold"],
                columns="ratio_name",
                values=metric,
                aggfunc="first",
            )

            ratios_here = [r for r in ratio_order_present if r in wide.columns]
            pair_results = []

            for ratio_a, ratio_b in combinations(ratios_here, 2):
                paired = wide[[ratio_a, ratio_b]].dropna()
                a = paired[ratio_a].to_numpy()
                b = paired[ratio_b].to_numpy()
                diff = a - b

                if len(paired) < 2:
                    stat, pval, status = np.nan, np.nan, "Insufficient paired observations"
                elif np.allclose(diff, 0):
                    stat, pval, status = 0.0, 1.0, "All paired differences are zero"
                else:
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            stat, pval = wilcoxon(
                                a, b,
                                alternative="two-sided",
                                zero_method="wilcox",
                                method="auto",
                            )
                        status = "ok"
                    except Exception as exc:
                        stat, pval, status = np.nan, np.nan, f"error: {type(exc).__name__}: {exc}"

                pair_results.append({
                    "sampler": sampler,
                    "model": model,
                    "metric": metric,
                    "metric_label": metric_label,
                    "ratio_a": ratio_a,
                    "ratio_b": ratio_b,
                    "n_paired": len(paired),
                    "mean_a": float(np.mean(a)) if len(a) else np.nan,
                    "mean_b": float(np.mean(b)) if len(b) else np.nan,
                    "mean_difference_a_minus_b": float(np.mean(diff)) if len(diff) else np.nan,
                    "wilcoxon_statistic": stat,
                    "p_value_raw": pval,
                    "status": status,
                })

            # Holm adjustment within this sampler/model/metric family
            valid_idx = [
                i for i, row in enumerate(pair_results)
                if pd.notna(row["p_value_raw"])
            ]
            if valid_idx:
                adjusted = multipletests(
                    [pair_results[i]["p_value_raw"] for i in valid_idx],
                    alpha=ALPHA,
                    method="holm",
                )[1]
                for i, p_adj in zip(valid_idx, adjusted):
                    pair_results[i]["p_value_holm"] = float(p_adj)

            for row in pair_results:
                row.setdefault("p_value_holm", np.nan)
                row["significant_holm"] = (
                    bool(row["p_value_holm"] < ALPHA)
                    if pd.notna(row["p_value_holm"]) else False
                )
                wilcoxon_rows.append(row)

wilcoxon_results = pd.DataFrame(wilcoxon_rows)
wilcoxon_results.to_csv(OUTPUT_DIR / "04_wilcoxon_pairwise_holm.csv", index=False)
display(wilcoxon_results)

## 6. Class-ratio effect plots for three metrics

In [ ]:

from scipy.stats import t
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# GRAPH: MEAN METRIC vs CLASS RATIO WITH 95% CI
# ============================================================

for sampler in samplers:
    sub = df[df["sampler"] == sampler].copy()

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(10, 15),
        sharex=True,
    )

    fig.suptitle(
        f"Effect of Class Ratio on Classification Performance ({sampler} + GA)",
        fontsize=16,
        fontweight="bold",
        y=0.995,
    )

    for metric_idx, (ax, (metric, metric_label)) in enumerate(
        zip(axes, METRICS.items())
    ):
        for model in models:
            model_data = sub[sub["model"] == model]

            stats = (
                model_data.groupby("ratio_name")[metric]
                .agg(["mean", "std", "count"])
                .reindex(ratio_order_present)
            )

            valid = stats["mean"].notna()
            if not valid.any():
                continue

            # Mean, SD, and number of observations
            means = stats.loc[valid, "mean"].to_numpy(dtype=float)
            stds = stats.loc[valid, "std"].to_numpy(dtype=float)
            counts = stats.loc[valid, "count"].to_numpy(dtype=int)

            # Standard error
            standard_errors = np.divide(
                stds,
                np.sqrt(counts),
                out=np.full_like(stds, np.nan, dtype=float),
                where=counts > 0,
            )

            # t critical value for a two-sided 95% confidence interval
            t_critical = np.array([
                t.ppf(0.975, df=n - 1) if n > 1 else np.nan
                for n in counts
            ])

            margin = t_critical * standard_errors

            ci_lower = means - margin
            ci_upper = means + margin

            # x positions follow the ratio order
            x_pos = np.array([
                ratio_order_present.index(r)
                for r in stats.index[valid]
            ])

            # Plot the mean and 95% confidence-interval error bars
            ax.errorbar(
                x_pos,
                means,
                yerr=np.vstack([
                    means - ci_lower,
                    ci_upper - means,
                ]),
                marker="o",
                linewidth=1.8,
                markersize=5,
                capsize=4,
                elinewidth=1.2,
                label=model,
            )

        ax.set_title(
            f"({chr(97 + metric_idx)}) {metric_label}",
            loc="left",
            fontweight="bold",
        )

        ax.set_ylabel(metric_label)
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.set_axisbelow(True)

        if metric in ["pr_auc_ap", "balanced_accuracy"]:
            ax.set_ylim(bottom=0)

        ax.legend(
            loc="best",
            frameon=False,
            ncol=min(4, max(1, len(models))),
        )

    axes[-1].set_xticks(np.arange(len(ratio_order_present)))
    axes[-1].set_xticklabels([
        RATIO_LABELS.get(r, r)
        for r in ratio_order_present
    ])
    axes[-1].set_xlabel("Target class ratio (majority:minority)")

    fig.tight_layout(
        rect=[0.03, 0.02, 0.98, 0.97],
        h_pad=2.0,
    )

    safe_sampler = "".join(
        ch if ch.isalnum() else "_"
        for ch in sampler
    )

    png_path = (
        OUTPUT_DIR
        / f"05_effect_of_class_ratio_{safe_sampler}_CI95.png"
    )
    pdf_path = (
        OUTPUT_DIR
        / f"05_effect_of_class_ratio_{safe_sampler}_CI95.pdf"
    )

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.show()

    print("Figures saved:")
    print(png_path)
    print(pdf_path)


## 7. Summary table for reporting

This table presents the mean ± SD for each model, ratio, and metric. The Friedman and Wilcoxon statistical tables are provided separately as CSV files.

In [ ]:
report_rows = []

for (sampler, model, ratio), group in df.groupby(
    ["sampler", "model", "ratio_name"], dropna=False
):
    row = {"sampler": sampler, "model": model, "ratio_name": ratio}
    for metric, metric_label in METRICS.items():
        vals = group[metric].dropna()
        row[f"{metric_label}_mean"] = vals.mean()
        row[f"{metric_label}_sd"] = vals.std(ddof=1)
        row[f"{metric_label}_n"] = vals.size
    report_rows.append(row)

report_table = pd.DataFrame(report_rows)
report_table["_ratio_order"] = report_table["ratio_name"].map(
    {r: i for i, r in enumerate(ratio_order_present)}
)
report_table = report_table.sort_values(
    ["sampler", "model", "_ratio_order"]
).drop(columns="_ratio_order")

report_table.to_csv(OUTPUT_DIR / "06_report_mean_sd.csv", index=False)
display(report_table)

print("\nOutputs saved to:", OUTPUT_DIR.resolve())
print("01_observation_audit.csv")
print("02_descriptive_statistics.csv")
print("03_friedman_tests.csv")
print("04_wilcoxon_pairwise_holm.csv")
print("05_effect_of_class_ratio_<sampler>.png / .pdf")
print("06_report_mean_sd.csv")